In [15]:
import pandas as pd
import numpy as np

In [16]:
np.random.seed(2006)

lista_32_paises = [
    "Alemania", "Costa Rica", "Polonia", "Ecuador",
    "Inglaterra", "Paraguay", "Trinidad y Tobago", "Suecia",
    "Argentina", "Costa de Marfil", "Países Bajos", "Serbia y Montenegro",
    "México", "Irán", "Angola", "Portugal",
    "Italia", "Ghana", "EEUU", "República Checa",
    "Brasil", "Croacia", "Australia", "Japón",
    "Francia", "Suiza", "Corea del Sur", "Togo",
    "España", "Ucrania", "Túnez", "Arabia Saudita"
]

paises_48_partidos = lista_32_paises * 3
np.random.shuffle(paises_48_partidos)
paises_48_partidos = paises_48_partidos[:48]

columnas_df1 = ["pais", "PTS_0", "PJ_0", "PG_0", "PP_0", "PE_0", "GF_0", "GC_0", "D_0"]
datos_df1 = {col: np.random.randint(1, 11, size=48) for col in columnas_df1[1:]}
datos_df1["pais"] = paises_48_partidos
df1 = pd.DataFrame(datos_df1)[columnas_df1]

columnas_df2 = ["pais", "VI", "SOT", "PKATT", "PKATTALLOW"]
datos_df2 = {col: np.random.randint(1, 11, size=32) for col in columnas_df2[1:]}
datos_df2["pais"] = lista_32_paises
df2 = pd.DataFrame(datos_df2)[columnas_df2]

In [17]:
trabajo = df1.merge(df2,on="pais",how="inner");print(trabajo)

                   pais  PTS_0  PJ_0  PG_0  PP_0  PE_0  GF_0  GC_0  D_0  VI  \
0          Países Bajos      7     5    10     4     5     6     9    4   3   
1             Australia     10     9     5     4     1     8     2    5   9   
2             Argentina      3     9     3     6     5     4    10    7   3   
3             Australia      2     7     8     1     4     4    10    9   9   
4                 Japón      8     4     9     6     5     1     6    7   6   
5         Corea del Sur      3    10     1     6    10     7    10    6   7   
6     Trinidad y Tobago      1     2     7     6     5     4     5    6   3   
7                  Irán      3     8     2     3    10     6     4    5   6   
8              Portugal      5     5     9     6     9     2     3    9   3   
9                Suecia     10     1     7     5     8     9     6    8   1   
10                Suiza      9     5     9     2     3    10    10   10   9   
11               México      3     3     1     8    

In [18]:
def calc_gkps(group):
    VI = group['VI'].sum()
    PJ_0 = group['PJ_0'].sum()
    GC_0 = group['GC_0'].sum()
    PKATTALLOW = group['PKATTALLOW'].sum() if 'PKATTALLOW' in group.columns else 0
    return ((VI/PJ_0)*60) + (40 - ((GC_0 - PKATTALLOW)/PJ_0)*10) if PJ_0 > 0 else 0

def calc_mds(group):
    VI = group['VI'].sum()
    PJ_0 = group['PJ_0'].sum()
    GC_0 = group['GC_0'].sum()
    PKATTALLOW = group['PKATTALLOW'].sum() if 'PKATTALLOW' in group.columns else 0
    return ((VI/PJ_0)*50) + (50 - (((GC_0 + PKATTALLOW)/PJ_0)*10)) if PJ_0 > 0 else 0

def calc_mos(group):
    GF_0 = group['GF_0'].sum()
    PJ_0 = group['PJ_0'].sum()
    SOT = group['SOT'].sum() if 'SOT' in group.columns else 0
    PKATT = group['PKATT'].sum() if 'PKATT' in group.columns else 0
    return ((GF_0/PJ_0)*0.5 + (SOT/PJ_0)*0.3 + ((GF_0 - PKATT)/SOT)*0.2) if PJ_0 > 0 and SOT > 0 else 0

def calc_mms(group):
    PTS_0 = group['PTS_0'].sum()
    PJ_0 = group['PJ_0'].sum()
    PG_0 = group['PG_0'].sum()
    SOT = group['SOT'].sum() if 'SOT' in group.columns else 0
    return (((PTS_0/(PJ_0*3))*40) + ((PG_0/PJ_0)*40) + ((SOT/PJ_0)*2)) if PJ_0 > 0 else 0

In [19]:
dfEspecializado = trabajo.groupby('pais').apply(
  lambda g: pd.Series({
      'gkps_0': calc_gkps(g),
      'mds_0': calc_mds(g),
      'mos_0': calc_mos(g),
      'mms_0': calc_mms(g)
  })
).reset_index()

/tmp/ipykernel_2500/2664494512.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dfEspecializado = trabajo.groupby('pais').apply(


In [20]:
trabajo.drop(columns=['VI', 'SOT', 'PKATT', 'PKATTALLOW'], inplace=True)
df_nuevo = trabajo.merge(dfEspecializado, on='pais', how='inner')
df_nuevo.rename(columns={'pais':'home'},inplace=True)

In [21]:
df_nuevo

,home,PTS_0,PJ_0,PG_0,PP_0,PE_0,GF_0,GC_0,D_0,gkps_0,mds_0,mos_0,mms_0
0,Países Bajos,7,5,10,4,5,6,9,4,72.000000,48.000000,0.893333,101.066667
1,Australia,10,9,5,4,1,8,2,5,115.000000,96.363636,0.139394,44.515152
2,Argentina,3,9,3,6,5,4,10,7,72.307692,50.000000,0.689231,50.769231
3,Australia,2,7,8,1,4,4,10,9,115.000000,96.363636,0.139394,44.515152
4,Japón,8,4,9,6,5,1,6,7,180.000000,118.000000,1.782857,168.266667
5,Corea del Sur,3,10,1,6,10,7,10,6,77.000000,70.000000,0.700000,9.000000
6,Trinidad y Tobago,1,2,7,6,5,4,5,6,135.000000,80.000000,2.716667,122.666667
7,Irán,3,8,2,3,10,6,4,5,88.750000,73.750000,0.812500,15.250000
8,Portugal,5,5,9,6,9,2,3,9,72.222222,62.222222,0.920833,83.555556
9,Suecia,10,1,7,5,8,9,6,8,54.000000,36.000000,1.320000,118.133333
